# Refusals in notes data
Notes column must contain any one of the possible ICD 9 and 10 vaccine refusal codes, start with "R/refused ", or contain a vaccine keywords and "R/refus"

In [ ]:
# setup
client = bq.Client(project='law-nero-phi-dho-scc-covid')

PROJECT_ID = "law-nero-phi-dho-scc-covid"
DATASET = "afc0125"
TABLE = "Notes"

# refusal codes
refusal_source_values = ['V64.00', 'V64.05', 'V64.09', 'V64.06','V64.07','Z28.1', 
                         'Z28.20', 'Z28.21', 'Z28.22', 'Z28.23', 'Z28.24', 'Z28.25',
                         'Z28.26', 'Z28.27', 'Z28.28', 'Z28.29', 
                         'Z28.82', 'Z28.83', 'Z28.89', 'Z28.9']
refusal_values_regex = '|'.join(refusal_source_values)

# vaccine keywords
vaccine_keywords = ['hpv', 'mmr', 'measle', 'papillo', 'hep', 'varicella', 
                    'menacwy', 'menb', 'mcv4', 'dtap', 'tdap']
vaccine_keywords_regex = '|'.join(vaccine_keywords)

# Use the regex in SQL
QUERY = f"""
SELECT patientuid, encounterdate, note
FROM `{PROJECT_ID}.{DATASET}.{TABLE}`
WHERE 
  LOWER(note) LIKE '%refus%' AND REGEXP_CONTAINS(LOWER(note), r'({vaccine_keywords_regex})')
  OR STARTS_WITH(LOWER(note), 'refused ')
  OR REGEXP_CONTAINS(note, r'({refusal_values_regex})')
"""
query_job = client.query(QUERY)
notes_refus = query_job.to_dataframe()

notes_refus.to_csv(f"/share/pi/deho-pi/AFC/BQ/Notes_Refusals.csv.gz", compression="gzip", escapechar='\\')